# La compuerta de integridad que respalda el corpus, y los controles que hacen que el motor se niegue

**Nivel 4: Más allá del estado del arte** · Requiere: 01-the-data/01-the-corpus-and-the-cases, 01-the-data/02-geometry-reconstruction · Tiempo de lectura: unos 25 minutos

## Objetivos

Una tabla copiada de un artículo es tan buena como la copia. Este cuaderno abre las dos salvaguardas que Fragmenta pone alrededor de sus datos: una compuerta que reproduce los estadísticos que imprime el propio artículo desde las filas incluidas cada vez que carga el corpus (afirmación BS-3), y un par de controles negativos que prueban que el motor se niega donde un número sería una invención (afirmación BS-2).

Al terminar habrá:

- recalculado el mínimo, el máximo, la media y la desviación estándar de las siete entradas y comparado con los valores que imprime el artículo;
- sembrado cuatro defectos de transcripción realistas en una copia del corpus y visto cuál de las dos capas de la compuerta atrapa cada uno;
- visto pasar el control negativo de geometría (seis tiros, sin escala) y el control negativo de diseño degenerado (seis diseños que no son tiros).

## Preparación

Instale los paquetes con los pines exactos de `notebooks/requirements.txt`. El motor es `blastfrag`, publicado en PyPI; los otros paquetes son su base numérica y la biblioteca de gráficos.

```bash
pip install blastfrag==0.4.0 numpy==2.5.4
```

Esta celda aún no importa nada del cuaderno: lee las versiones instaladas desde los metadatos de los paquetes y las imprime, de modo que la salida registra qué motor produjo cada número que sigue.

In [1]:
# Imprima la versión instalada de cada paquete que usa este cuaderno
from importlib.metadata import version

for name in ("blastfrag", "numpy",):
    print(f"{name}=={version(name)}")

blastfrag==0.4.0
numpy==2.5.4


## Datos

| Fuente | Titular y URL | Clase de licencia | Archivo que lee el cuaderno |
|---|---|---|---|
| Las 97 filas y los estadísticos descriptivos impresos | Hudaverdi, Kulatilake y Kuzu 2010, https://doi.org/10.1002/nag.957 | hechos numéricos reutilizados con cita; el artículo nunca se distribuye | la tabla y `PUBLISHED_DESCRIPTIVE_STATS` dentro del paquete `blastfrag` |
| El control negativo de geometría (Miami) | este repositorio (MIT), escrito por la ejecución offline | la licencia propia del producto | `data/derived/real-miami/case.json` |
| El control negativo de diseño degenerado | este repositorio (MIT), escrito por la ejecución offline | la licencia propia del producto | `data/derived/ctrl-degenerate/case.json` |

## Método

La compuerta tiene dos capas independientes, y responden preguntas distintas.

1. **Los estadísticos impresos.** El artículo imprime un mínimo, un máximo, una media y una desviación estándar para cada una de las siete entradas. La compuerta recalcula los cuatro desde las filas incluidas y compara cada uno con el valor impreso *a la precisión con que se imprimió*. Una discrepancia significa que el archivo ya no es la tabla publicada.
2. **El resumen de contenido.** Los estadísticos detectan mal el cambio de una sola celda: corregir un factor de carga mueve una media de 97 filas en menos que un paso de redondeo. Un resumen SHA-256 de toda la tabla, fijado en el motor, atrapa cualquier cambio.

Ambas corren dentro de `load_training_corpus()`. La primera celda llama a la compuerta directamente y pone los estadísticos recalculados junto a los impresos.

In [2]:
# Importaciones, y luego la compuerta sobre el corpus incluido, con los estadísticos recalculados junto a los impresos
import dataclasses
import json
import math
from pathlib import Path

import blastfrag as bf

DERIVED = Path("../../../data/derived")
train = bf.load_training_corpus()
bf.check_source_integrity(train)
print("compuerta sobre el corpus incluido: aprobada")

print(f"{'entrada':<10}{'estadístico':<14}{'artículo imprime':>16}{'recalculado':>14}")
for name, printed in bf.PUBLISHED_DESCRIPTIVE_STATS.items():
    values = [getattr(b, name) for b in train]
    mean = sum(values) / len(values)
    sd = math.sqrt(sum((v - mean) ** 2 for v in values) / (len(values) - 1))
    for label, want, got in (("min", printed.minimum, min(values)), ("max", printed.maximum, max(values)),
                              ("media", printed.mean, mean), ("sd", printed.sd, sd)):
        print(f"{name:<10}{label:<14}{want:>16}{got:>14.4f}")

compuerta sobre el corpus incluido: aprobada
entrada   estadístico   artículo imprime   recalculado
S_over_B  min                        1.0        1.0000
S_over_B  max                       1.75        1.7500
S_over_B  media                     1.19        1.1890
S_over_B  sd                       0.117        0.1168
H_over_B  min                       1.33        1.3300
H_over_B  max                       6.82        6.8200
H_over_B  media                     3.34        3.3452
H_over_B  sd                       1.634        1.6337
B_over_D  min                      17.98       17.9800
B_over_D  max                      39.47       39.4700
B_over_D  media                    27.35       27.3549
B_over_D  sd                       4.838        4.8386
T_over_B  min                        0.5        0.5000
T_over_B  max                       4.67        4.6700
T_over_B  media                     1.26        1.2628
T_over_B  sd                       0.674        0.6739
Pf_kg_m3  min       

## Estado del arte

Tres hechos de la base de investigación del producto (el SDD) enmarcan lo que ya se sabe.

- El corpus es una tabla publicada. Hudaverdi, Kulatilake y Kuzu 2010 (https://doi.org/10.1002/nag.957) la imprimen junto con sus propios estadísticos resumen, lo que hace posible una verificación independiente.
- La base de investigación del SDD registra que una copia de la tabla guardada en la bóveda del laboratorio tenía cinco defectos de transcripción, dos de ellos en la variable objetivo de la regresión, encontrados al comparar cada celda con las tablas del artículo. Un estudio que reutiliza una tabla copiada sin esa comparación hereda esos defectos en silencio.
- Los estudios que ajustan predictores aprendidos a estas filas (Amoako, Jha y Zhong 2022, https://doi.org/10.3390/mining2020013; Sui et al. 2025, https://doi.org/10.3390/app15031254) reportan predictores de las siete entradas. Un regresor entrenado devuelve un número para cualquier vector de entrada; no tiene un dominio declarado en el que se niegue a responder. Si el volumen de roca y la carga pueden conocerse para un tiro queda fuera de lo que un predictor así puede expresar.

## Más allá del estado del arte

**Afirmación BS-3, la compuerta de integridad al cargar.** Cada carga del corpus reproduce los estadísticos impresos del artículo y un resumen fijado. La celda siguiente siembra cuatro defectos en copias del corpus y ejecuta la compuerta sobre cada una, primero solo con los estadísticos (`check_digest=False`) y luego con ambas capas. El primer defecto es el histórico por el que se escribió la compuerta: un máximo de factor de carga de 1,47 donde el artículo imprime 1,26.

In [3]:
# Sembrar un defecto de transcripción a la vez en una copia del corpus y ejecutar ambas capas de la compuerta
peak = max(train, key=lambda b: b.Pf_kg_m3)
first = train[0]
defects = {
    "máximo de factor de carga 1,26 digitado como 1,47": (peak.blast_id, {"Pf_kg_m3": 1.47}),
    "módulo 60,0 digitado como 6,0": (first.blast_id, {"E_GPa": 6.0}),
    "razón de taco 0,78 digitada como 0,79": (first.blast_id, {"T_over_B": 0.79}),
    "tamaño medido 0,37 digitado como 0,73": (first.blast_id, {"x50_m": 0.73}),
}

def verdict(rows, with_digest):
    # Verdadero cuando la compuerta lanza una excepción, que es el resultado buscado para un defecto sembrado
    try:
        bf.check_source_integrity(rows, check_digest=with_digest)
        return False
    except bf.ContractViolation:
        return True

print(f"{'defecto sembrado':<62}{'estadísticos':>14}{'estadísticos + resumen':>26}")
for label, (blast_id, change) in defects.items():
    corrupted = [dataclasses.replace(b, **change) if b.blast_id == blast_id else b for b in train]
    print(f"{label:<62}{'atrapado' if verdict(corrupted, False) else 'no detectado':>14}"
          f"{'atrapado' if verdict(corrupted, True) else 'no detectado':>26}")

defecto sembrado                                                estadísticos    estadísticos + resumen
máximo de factor de carga 1,26 digitado como 1,47                   atrapado                  atrapado
módulo 60,0 digitado como 6,0                                       atrapado                  atrapado
razón de taco 0,78 digitada como 0,79                           no detectado                  atrapado
tamaño medido 0,37 digitado como 0,73                           no detectado                  atrapado


**Afirmación BS-2, la abstención como resultado de primera clase.** Un brazo que necesita un volumen de roca no debe responder donde no existe. El control de geometría son los seis tiros de Miami. La celda lee el artefacto versionado de ese caso y cuenta, para cada uno de los 14 brazos, cuántos de los seis tiros respondió y cuántos rechazó. Los brazos clásicos deben rechazar los seis; los brazos que usan solo las razones pueden responder.

In [4]:
# Contar tiros respondidos y rechazados por brazo en el artefacto versionado de Miami
miami = json.loads((DERIVED / "real-miami" / "case.json").read_text(encoding="utf-8"))
print(f"{'brazo':<24}{'respondió':>10}{'rechazó':>10}")
for arm, per_blast in sorted(miami["predictions"].items()):
    refused = sum(1 for p in per_blast.values() if p["abstained"])
    print(f"{arm:<24}{len(per_blast) - refused:>10}{refused:>10}")
print("control registrado por la ejecución offline:", miami["controls"]["geometry_negative_control"])

# La misma negativa, en vivo desde el motor instalado
blast = next(b for b in train if b.site == "Miami")
live = bf.Kuznetsov().predict_one(blast)
print("motor en vivo, Kuznetsov, tiro", blast.blast_id, "->", live.x50_m, "|", live.abstain_reason[:60], "...")

brazo                    respondió   rechazó
crush-zone                       0         6
group-discriminant               0         6
kuz-ram                          0         6
kuznetsov                        0         6
kuznetsov-capped                 0         6
kuznetsov-transfer               0         6
published-neural-net             6         0
published-regression             6         0
random-forest                    6         0
refitted-regression              6         0
stacking                         6         0
svr-rbf                          6         0
swebrec                          0         6
xgboost                          6         0
control registrado por la ejecución offline: {'n_answered_anyway': 0, 'n_without_geometry': 6, 'passed': True}
motor en vivo, Kuznetsov, tiro Mi1 -> None | Miami publishes no hole diameter, so rock volume and charge  ...


El segundo control es un conjunto de seis diseños que no son tiros: un taco más largo que el banco, de modo que no hay columna de carga. Un modelo basado solo en razones los valoraría sin objeción. El producto protege el diseño antes de que corra cualquier brazo, así que todos los brazos deben rechazar todos los diseños.

In [5]:
# Para cada brazo contar los diseños degenerados que respondió; el control pasa cuando el total es cero
degenerate = json.loads((DERIVED / "ctrl-degenerate" / "case.json").read_text(encoding="utf-8"))
answered = {
    arm: sum(1 for p in per_blast.values() if not p["abstained"])
    for arm, per_blast in degenerate["predictions"].items()
}
print("brazos revisados:", len(answered), "| diseños respondidos por algún brazo:", sum(answered.values()))
print("control registrado por la ejecución offline:", degenerate["controls"]["degenerate_negative_control"])
first_design = degenerate["blasts"][0]
print(first_design["blast_id"], "razón (en):", first_design["degenerate_reason"]["en"])
print(first_design["blast_id"], "razón (es):", first_design["degenerate_reason"]["es"])

brazos revisados: 14 | diseños respondidos por algún brazo: 0
control registrado por la ejecución offline: {'answered_by': [], 'n_answered_anyway': 0, 'n_degenerate': 6, 'passed': True}
DG01 razón (en): the stemming is 4.60 burdens in a bench 1.40 burdens tall, so there is no charge column. This is not a blast.
DG01 razón (es): el taco mide 4,60 bordos en un banco de 1,40 bordos de alto, así que no hay columna de carga. Esto no es una voladura.


## Resultados

**La compuerta mantiene el corpus fiel al artículo.** Sobre las filas incluidas, el mínimo, el máximo, la media y la desviación estándar recalculados de las siete entradas coinciden con los valores impresos a la precisión que usó el artículo, y el resumen coincide.

**Las dos capas no son redundantes.** El máximo del factor de carga digitado como 1,47 y el módulo digitado como 6,0 mueven un extremo impreso, y la capa de estadísticos los atrapa. La razón de taco cambiada en el segundo decimal de una fila, y el tamaño medido transpuesto, no mueven ningún estadístico impreso de forma visible (el tamaño medido ni siquiera está entre los estadísticos impresos), así que solo el resumen los atrapa. El cuaderno muestra cada resultado en la tabla anterior en vez de afirmarlo. Una tabla copiada verificada solo contra estadísticos resumen dejaría pasar un defecto así.

**Las negativas funcionan como controles.** En el artefacto de Miami los brazos que necesitan un volumen de roca no responden ninguno de los seis tiros, mientras que los brazos que usan solo las razones responden los seis (el discriminante de grupo devuelve un grupo de rigidez de la roca, no un tamaño, así que aparece rechazando en todas las filas): el motor no oculta la diferencia, la informa. En el caso degenerado ningún brazo responde ninguno de los seis diseños, y cada negativa se explica a sí misma en inglés y en español.

**Qué no muestra esto.** Pasar la compuerta muestra que el archivo es la tabla publicada; no muestra que la tabla publicada sea correcta respecto de los tiros. Y una negativa no es prueba de corrección en otra parte: los controles muestran que el motor se niega donde debe, no que sea exacto donde responde. Esa pregunta es el tema de los temas 04 y 05.

## Resumen

- Cargar el corpus reproduce los 28 estadísticos impresos del artículo (4 por cada una de las 7 entradas) y un resumen fijado; los estadísticos atrapan cambios en un extremo y el resumen atrapa el resto (afirmación BS-3).
- Dos controles negativos se distribuyen como datos: seis tiros sin escala y seis diseños que no son tiros. El motor rechaza los doce donde debe, con razones en ambos idiomas (afirmación BS-2).
- Siguiente: el tema 02 usa la geometría reconstruida para ejecutar la ecuación clásica.

## Referencias

- Hudaverdi, Kulatilake y Kuzu 2010, https://doi.org/10.1002/nag.957 (la tabla, sus estadísticos impresos, los cinco defectos hallados por comparación)
- Amoako, Jha y Zhong 2022, https://doi.org/10.3390/mining2020013 (predictores aprendidos sobre las mismas filas)
- Sui, Zhou, Zhao, Yang y Zou 2025, https://doi.org/10.3390/app15031254 (el conjunto de apilamiento y los tiros de campo)
- `blastfrag` 0.4.0, el motor, en PyPI: https://pypi.org/project/blastfrag/